<a href="https://colab.research.google.com/github/anua74999-dot/anu/blob/master/Task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import time
import threading
import ipywidgets as widgets
from IPython.display import display, clear_output

In [7]:
running = False
start_time = 0
elapsed = 0
laps = []
title = widgets.HTML("""
<div style='text-align:center;padding:10px 0;'>
<h1 style='margin:0;font-size:44px;font-family:Segoe UI;background:linear-gradient(90deg,#06b6d4,#2563eb,#8b5cf6);-webkit-background-clip:text;-webkit-text-fill-color:transparent;'>⏱ Elite Stopwatch</h1>
<p style='margin:6px 0;color:#64748b;'>Beautiful timing experience for Google Colab</p>
</div>
""")

def dial_html(text):
    return f"""
    <div style='margin:10px auto;width:330px;height:330px;border-radius:50%;background:conic-gradient(#06b6d4,#2563eb,#8b5cf6,#06b6d4);padding:10px;box-shadow:0 25px 60px rgba(37,99,235,.28);'>
      <div style='width:100%;height:100%;border-radius:50%;background:radial-gradient(circle,#ffffff,#eef2ff);display:flex;align-items:center;justify-content:center;border:8px solid rgba(255,255,255,.65);'>
        <div style='text-align:center;'>
          <div style='font-size:18px;color:#64748b;'>TIME</div>
          <div style='font-size:44px;font-weight:700;color:#111827;'>{text}</div>
        </div>
      </div>
    </div>
    """

display_box = widgets.HTML(value=dial_html('00:00:00.00'))
status = widgets.HTML("<p style='text-align:center;color:#64748b;'>Ready to start</p>")

lap_output = widgets.Output(layout={
    'border':'1px solid #dbeafe',
    'height':'230px',
    'overflow_y':'auto',
    'padding':'12px',
    'border_radius':'18px'
})

# Buttons
start_btn = widgets.Button(description='Start', icon='play', button_style='success')
pause_btn = widgets.Button(description='Pause', icon='pause', button_style='warning')
lap_btn = widgets.Button(description='Lap', icon='flag', button_style='info')
reset_btn = widgets.Button(description='Reset', icon='refresh', button_style='danger')

for b in [start_btn,pause_btn,lap_btn,reset_btn]:
    b.layout.width='135px'
    b.layout.height='48px'

# ---------- Logic ----------
def format_time(sec):
    hrs = int(sec // 3600)
    mins = int((sec % 3600) // 60)
    secs = int(sec % 60)
    cs = int((sec - int(sec)) * 100)
    return f"{hrs:02}:{mins:02}:{secs:02}.{cs:02}"


def update_ui(t):
    display_box.value = dial_html(format_time(t))


def updater():
    global running, elapsed
    while running:
        current = elapsed + (time.time() - start_time)
        update_ui(current)
        time.sleep(0.03)


def start_clicked(btn):
    global running, start_time
    if not running:
        running = True
        start_time = time.time()
        status.value = "<p style='text-align:center;color:#16a34a;'>Running...</p>"
        threading.Thread(target=updater, daemon=True).start()


def pause_clicked(btn):
    global running, elapsed
    if running:
        elapsed += time.time() - start_time
        running = False
        status.value = "<p style='text-align:center;color:#d97706;'>Paused</p>"


def reset_clicked(btn):
    global running, elapsed, laps
    running = False
    elapsed = 0
    laps = []
    update_ui(0)
    status.value = "<p style='text-align:center;color:#dc2626;'>Reset completed</p>"
    with lap_output:
        clear_output()


def lap_clicked(btn):
    current = elapsed + (time.time() - start_time if running else 0)
    laps.append(current)
    with lap_output:
        clear_output()
        print('✨ Lap Records')
        for i,t in enumerate(laps,1):
            print(f"Lap {i:02}   {format_time(t)}")

# Bind events
start_btn.on_click(start_clicked)
pause_btn.on_click(pause_clicked)
lap_btn.on_click(lap_clicked)
reset_btn.on_click(reset_clicked)

# Layout
buttons = widgets.HBox([start_btn,pause_btn,lap_btn,reset_btn], layout={'justify_content':'center','gap':'12px'})
section = widgets.HTML("<h3 style='text-align:center;color:#1e3a8a;'>Lap History</h3>")
footer = widgets.HTML("<center><p style='color:#64748b;font-style:italic;'>Every second counts.</p></center>")

card = widgets.VBox([title, display_box, status, buttons, section, lap_output, footer])
card.layout.width='780px'
card.layout.margin='30px auto'
card.layout.padding='28px'
card.layout.border_radius='32px'
card.layout.box_shadow='0 25px 60px rgba(0,0,0,.16)'
card.layout.background='linear-gradient(135deg,#eff6ff,#ffffff,#dbeafe,#eef2ff)'

# Render
display(card)